# Phase 0 — Chạy thử train SFT

Kiểm tra hạ tầng train trước Phase 1: train vài step trên 4 ví dụ viết tay, lưu checkpoint đầy đủ, đẩy lên HF Hub, rồi resume từ Hub. Đây **không** phải thí nghiệm; loss ở đây không mang ý nghĩa năng lực.

Nên dùng runtime **T4 GPU** (Runtime → Change runtime type) để đo thời gian và bộ nhớ như lúc train thật. Logic train nằm trong `src/train.py`; notebook chỉ gọi nó.


## 1. Cài thư viện
Dùng torch có sẵn của Colab; cài đúng các phiên bản trong `requirements.txt`.

In [ ]:
%pip uninstall -q -y torchao
%pip install -q transformers==5.20.2 huggingface-hub==1.32.0 psutil==7.2.2 safetensors==0.8.0 pyyaml==6.0.3 accelerate==1.15.0 datasets==5.1.0 peft==0.21.2 trl==1.16.0
from importlib.metadata import PackageNotFoundError, version
wanted = {"transformers": "5.20.2", "huggingface-hub": "1.32.0", "psutil": "7.2.2", "safetensors": "0.8.0", "pyyaml": "6.0.3", "accelerate": "1.15.0", "datasets": "5.1.0", "peft": "0.21.2", "trl": "1.16.0"}
installed = {name: version(name) for name in wanted}
print(installed)
assert installed == wanted, "Phiên bản chưa khớp requirements.txt; chạy lại cell này."
try:
    raise AssertionError(f"torchao {version('torchao')} vẫn còn; chạy lại cell này.")
except PackageNotFoundError:
    pass


## 2. Nạp mã nguồn của dự án
Sửa logic tại repo rồi chạy lại `python scripts/build_playground_notebook.py` để cập nhật notebook.

In [ ]:
import base64, hashlib, json, os, sys
from pathlib import Path

payload = base64.b64decode("...")
assert hashlib.sha256(payload).hexdigest() == "6bd5ec756c3cec16c164655d866c82eb4ff1db6a2e967573113c40bfea16bdb4"
root = Path("/content/sft-base-playground")
root.mkdir(parents=True, exist_ok=True)
for relative, source in json.loads(payload).items():
    target = root / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(source, encoding="utf-8")
os.chdir(root)
sys.path.insert(0, str(root))
# One GPU only (Kaggle gives two T4s): with two, Trainer splits each batch across them and the effective batch doubles.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
# Các object đã tạo (như lab) vẫn giữ nguyên; lần import tiếp dùng source mới.
for module_name in list(sys.modules):
    if module_name == "src" or module_name.startswith("src."):
        del sys.modules[module_name]
print("Source bundle SHA-256:", "6bd5ec756c3cec16c164655d866c82eb4ff1db6a2e967573113c40bfea16bdb4")
print("Working directory:", root)


## 3. Kiểm tra mã
Gồm test resume trên Llama ngẫu nhiên cực nhỏ: chạy liền 4 step phải cho cùng loss với 2 step, lưu, resume, 2 step.

In [ ]:
import subprocess
checks = subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"], capture_output=True, text=True)
print(checks.stdout)
print(checks.stderr)
checks.check_returncode()


## 4. Đăng nhập HF Hub

Cần secret `HF_TOKEN` (token có quyền ghi) trong 🔑 **Secrets** của Colab, đã bật **Notebook access**. Token không được in ra hay ghi vào repo. Điền `HUB_MODEL_ID` dạng `<username>/<tên repo>`; repo được tạo ở chế độ private ở lần push đầu.


In [ ]:
from google.colab import userdata
from huggingface_hub import whoami

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
HUB_MODEL_ID = ""  # ví dụ "your-username/smollm2-360m-sft-smoke"
print("Logged in as:", whoami()["name"])
assert HUB_MODEL_ID, "Điền HUB_MODEL_ID trước khi chạy tiếp."


## 5. Chạy liền 5 step (mốc so sánh)

Không đẩy lên Hub. Đường loss này dùng để so với lần chạy bị ngắt rồi resume ở mục 6–7.


In [ ]:
import json, shutil, subprocess

shutil.rmtree("results/runs", ignore_errors=True)  # Mỗi lần chạy thử bắt đầu sạch.

def train(*extra):
    command = [sys.executable, "-m", "src.train", "--config", "configs/smoke.yaml", *extra]
    completed = subprocess.run(command, capture_output=True, text=True)
    print(completed.stdout[-3000:])
    if completed.returncode:
        print(completed.stderr[-5000:])
    completed.check_returncode()

def losses(output_dir):
    history = json.loads(Path(output_dir, "metrics.json").read_text())["log_history"]
    return {row["step"]: (round(row["loss"], 5), row["assistant_tokens_seen"]) for row in history if "loss" in row}

train("--output-dir", "results/runs/smoke-straight")
straight = losses("results/runs/smoke-straight")
print(straight)


## 6. Chạy lại, dừng sau step 2 và đẩy checkpoint lên Hub
Checkpoint ở step 2 được đẩy lên ô `checkpoint-a/` của repo Hub (`resume.json` trỏ tới ô mới nhất).

In [ ]:
train("--output-dir", "results/runs/smoke-split", "--hub-model-id", HUB_MODEL_ID, "--stop-after", "2")
print(losses("results/runs/smoke-split"))


## 7. Resume từ Hub và so sánh
Tải checkpoint mới nhất (theo `resume.json`) từ Hub, train tiếp tới step 5. Loss ở step 3–5 phải gần bằng lần chạy liền (fp16 trên GPU có thể lệch ở chữ số cuối).

In [ ]:
train("--output-dir", "results/runs/smoke-split", "--hub-model-id", HUB_MODEL_ID, "--resume", "hub")
resumed = losses("results/runs/smoke-split")
for step in sorted(straight):
    print(step, "straight", straight[step], "resumed", resumed.get(step))
metrics = json.loads(Path("results/runs/smoke-split/metrics.json").read_text())
print({key: metrics[key] for key in ("global_step", "assistant_tokens_seen", "train_runtime", "peak_gpu_memory_gib")})


## 8. Ghi lại môi trường
In danh sách phiên bản thư viện để lưu vào `requirements.lock` trong repo.

In [ ]:
print(subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout)
